# 04 KR2036 驗證：需求、裝置容量、發電佔比

改寫自上游 `notebooks/validation/validation_nigeria.ipynb`（第 2–4 章）與
`validation_namibia.ipynb`（第 2–3 章）。

**所有對照值一律從 repo 既有檔案讀取，本 notebook 不從記憶或網路填任何數字。**
資料來源如下：

| 對照值 | 來源檔案 |
|---|---|
| 第 10 次電力供需基本計畫（BPLE）2036 容量目標 | `data/agg_p_nom_minmax_KR2036_BPLE.csv`（模型實際套用的上下限）|
| BPLE 官方值、各情境彙整 | `results/compare/scenario_table.csv`（`BPLE官方` 欄）|
| 論文 Table 2（容量／CAPEX） | `results/compare/kwak_full_vs_paper/table2_capacity_capex.csv` |
| 論文 Table 3（發電量／OPEX） | `results/compare/kwak_full_vs_paper/table3_generation_opex.csv` |
| 假設對照 | `results/compare/kwak_full_vs_paper/table0_assumptions.csv` |
| 文字歸因 | `docs/比較分析_kwak_full對論文_最終版.md` |

**與原版的主要差異**

| 項目 | 上游原版 | 韓國版 |
|---|---|---|
| 外部驗證來源 | IRENA / USAID / Our World in Data 三個外部來源 | **全部移除**，改用 repo 內的 BPLE 與論文對照值 |
| 需求換算 | `sum() × 8760 / len(snapshots)` 手動補償代表日 | 直接用 `snapshot_weightings`，不需手動換算 |
| 國家過濾 | `loads_t.p.filter(regex="NG *")` / `"NA *"` | 不過濾（全網只有 KR）|
| 容量欄位 | 只看 `p_nom` | `p_nom`（既有）與 `p_nom_opt`（最佳化後）並列 |

## 0. 環境設定與對照值載入

In [ ]:
import sys
import warnings
from pathlib import Path

_here = Path.cwd()
for cand in (_here, _here / "notebooks_kr", _here.parent):
    if (cand / "_kr_common.py").exists():
        sys.path.insert(0, str(cand))
        break

import _kr_common as K

K.setup_matplotlib()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

PAPER_DIR = K.ROOT / "results/compare/kwak_full_vs_paper"
REF = {
    "BPLE 限制檔": K.ROOT / "data/agg_p_nom_minmax_KR2036_BPLE.csv",
    "情境彙整表": K.ROOT / "results/compare/scenario_table.csv",
    "論文 Table 2": PAPER_DIR / "table2_capacity_capex.csv",
    "論文 Table 3": PAPER_DIR / "table3_generation_opex.csv",
    "假設對照 Table 0": PAPER_DIR / "table0_assumptions.csv",
    "文字歸因": K.ROOT / "docs/比較分析_kwak_full對論文_最終版.md",
}
missing = [k for k, v in REF.items() if not v.exists()]
if missing:
    raise FileNotFoundError(f"缺少對照檔案，請先補齊再跑：{missing}")
for k, v in REF.items():
    print(f"✓ {k:14s} {v.relative_to(K.ROOT)}")

In [ ]:
bple_limits = pd.read_csv(REF["BPLE 限制檔"], skiprows=1)
scenario_table = pd.read_csv(REF["情境彙整表"], index_col=0, encoding="utf-8-sig")
t0 = pd.read_csv(REF["假設對照 Table 0"], index_col=0, encoding="utf-8-sig")
t2 = pd.read_csv(REF["論文 Table 2"], index_col=0, encoding="utf-8-sig")
t3 = pd.read_csv(REF["論文 Table 3"], index_col=0, encoding="utf-8-sig")

n_base = K.load_network("baseline")
n_kwak = K.load_network("kwak_full")
NETS = {"baseline": n_base, "kwak_full": n_kwak}

print("BPLE 模型實際套用的容量上下限 [MW]：")
bple_limits

## 2. 需求驗證

上游原版用 `sum() × 8760 / len(snapshots)` 手動補償代表日，我們的網路有
完整的 `snapshot_weightings`（3H 權重為 3.0），直接加權即可，不需要手動換算。

對照值：
- **BPLE 2036 目標 667.3 TWh**——基準情境即以此校準（見 `config.yaml` 的
  `demand` 區塊註解 `667_300_000 MWh <- 第10次電기본 2036 發電량 667.3 TWh`）。
- **論文 707 TWh**——來自 `table0_assumptions.csv` 的「需求」列。

In [ ]:
print("table0 的需求列（論文對照值來源）：")
print(t0.loc[["需求"]].to_string())

In [ ]:
rows = []
for key, m in NETS.items():
    w = m.snapshot_weightings.generators
    load = m.loads_t.p_set.sum(axis=1)
    rows.append({
        "情境": K.SCENARIOS[key]["label"],
        "年需求_TWh": float(load.mul(w).sum() / 1e6),
        "尖峰負載_GW": float(load.max() / 1e3),
        "最低負載_GW": float(load.min() / 1e3),
        "負載因數": float(load.mean() / load.max()),
    })
demand = pd.DataFrame(rows).set_index("情境")
demand["對 BPLE 667.3 TWh 偏差%"] = (demand["年需求_TWh"] / 667.3 - 1) * 100
demand["對論文 707 TWh 偏差%"] = (demand["年需求_TWh"] / 707.0 - 1) * 100

K.FIG_DIR.mkdir(parents=True, exist_ok=True)
demand.round(3).to_csv(K.FIG_DIR / "04_table_demand_validation.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "04_table_demand_validation.csv")
demand.round(2)

### 2.1 需求驗證結論

兩個情境**各自對準不同的需求假設**，這是設計上的選擇而不是誤差：

- **基準情境 667.3 TWh** 對準 BPLE 2036 官方發電量目標，偏差 0.0%。
- **kwak_full 707.0 TWh** 對準論文假設（2023 年用電量以 1.7%/yr 外推），偏差 0.0%。

兩者相差 39.7 TWh（約 6%），在比較兩情境的任何結果時都必須把這一點納入考量
——這也是 02 第 7.3 節必須另外建立受控對照組的原因。

## 3. 裝置容量驗證

比較三組數字：模型的 `p_nom`（既有）與 `p_nom_opt`（最佳化後）、
BPLE 官方目標、論文 Table 2。

**先說明一個 repo 內部的不一致**：`scenario_table.csv` 的 `BPLE官方` 欄與
`agg_p_nom_minmax_KR2036_BPLE.csv`（模型實際套用的限制）在兩個項目上對不起來，
下面直接把兩者並列，不擅自選一個。

In [ ]:
# 模型實際套用的 BPLE 限制（MW → GW）
lim = bple_limits.copy()
lim.columns = ["country", "carrier", "min", "max"]
lim["min_GW"] = lim["min"] / 1e3
lim["max_GW"] = lim["max"] / 1e3

# scenario_table 的 BPLE官方 欄
cap_rows = [i for i in scenario_table.index if i.endswith("容量 GW")]
bple_col = scenario_table.loc[cap_rows, "BPLE官方"]
bple_col.index = [i.replace(" 容量 GW", "") for i in bple_col.index]

compare_src = pd.DataFrame({
    "scenario_table 的 BPLE官方_GW": bple_col,
    "限制檔 min_GW": lim.set_index("carrier")["min_GW"],
    "限制檔 max_GW": lim.set_index("carrier")["max_GW"],
})
compare_src = compare_src.dropna(how="all")
compare_src

兩處不一致（**不是我算錯，是 repo 內兩份檔案本身不同**）：

- **CCGT**：`scenario_table` 的 `BPLE官方` 寫 **62.9 GW**，但限制檔與 `config.yaml`
  註解都是 **64.6 GW**（`LNG 64,600`）。62.9 其實是**論文**的值，
  `scenario_table` 的 BPLE 欄在這一格很可能誤填成論文值。
- **biomass**：`scenario_table` 寫 **1.8 GW**，限制檔是 **1.62 GW**。

下面的驗證表以**限制檔**為準（那才是模型實際套用的），並把 `scenario_table`
的值另列一欄供對照。

In [ ]:
# BPLE 官方目標 = 限制檔的區間中點（固定值時 min == max）。
# 驗證：solar (59.13+72.27)/2 = 65.7、wind (30.68+37.498)/2 = 34.089，
# 與 scenario_table 的 BPLE官方 欄一致，證明這個推導方式正確；
# 不一致的 CCGT 與 biomass 以限制檔為準（那才是模型實際套用的）。
bple_target = ((lim["min_GW"] + lim["max_GW"]) / 2)
bple_target.index = lim["carrier"]
bple_target["wind合計"] = bple_target.pop("onwind+offwind-ac+offwind-dc")
print("由限制檔推導的 BPLE 官方目標 [GW]：")
print(bple_target.round(3).to_string())

CARRIER_MAP = {
    "solar": "solar",
    "onwind": "onwind",
    "offwind-ac": "offwind-ac",
    "offwind-dc": "offwind-dc",
    "CCGT": "CCGT",
    "nuclear": "nuclear",
    "coal": "coal",
    "biomass": "biomass",
}

rows = []
for carrier in CARRIER_MAP:
    r = {"carrier": carrier, "中文": K.carrier_label(carrier)}
    for key, m in NETS.items():
        g = m.generators[m.generators.carrier == carrier]
        r[f"{K.SCENARIOS[key]['label']} p_nom_GW"] = float(g.p_nom.sum() / 1e3)
        r[f"{K.SCENARIOS[key]['label']} p_nom_opt_GW"] = float(g.p_nom_opt.sum() / 1e3)
    paper_key = "offwind" if carrier.startswith("offwind") else carrier
    r["論文 Table2_GW"] = (
        float(t2.at[paper_key, "paper_capacity_MW"]) / 1e3 if paper_key in t2.index else np.nan
    )
    r["BPLE 官方_GW（限制檔）"] = float(bple_target.get(carrier, np.nan))
    r["BPLE_GW（scenario_table）"] = float(bple_col.get(carrier, np.nan))
    rows.append(r)

cap = pd.DataFrame(rows).set_index("carrier")

# 風電三項在 BPLE 與論文都是合計，另外補一列
for key, m in NETS.items():
    g = m.generators[m.generators.carrier.isin(["onwind", "offwind-ac", "offwind-dc"])]
    cap.loc["wind合計", f"{K.SCENARIOS[key]['label']} p_nom_GW"] = float(g.p_nom.sum() / 1e3)
    cap.loc["wind合計", f"{K.SCENARIOS[key]['label']} p_nom_opt_GW"] = float(g.p_nom_opt.sum() / 1e3)
cap.loc["wind合計", "中文"] = "風電合計"
cap.loc["wind合計", "BPLE 官方_GW（限制檔）"] = float(bple_target.get("wind合計", np.nan))
cap.loc["wind合計", "BPLE_GW（scenario_table）"] = float(bple_col.get("wind合計", np.nan))
cap.loc["wind合計", "論文 Table2_GW"] = (
    float(t2.at["onwind", "paper_capacity_MW"]) + float(t2.at["offwind", "paper_capacity_MW"])
) / 1e3

cap.round(3).to_csv(K.FIG_DIR / "04_table_capacity_validation.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "04_table_capacity_validation.csv")
cap.round(2)

In [ ]:
plot_rows = ["nuclear", "coal", "CCGT", "biomass", "solar", "wind合計"]
pc = cap.loc[plot_rows]
series = [
    ("基準情境 p_nom_opt_GW", "基準情境（最佳化後）", "#4c72b0"),
    ("kwak_full p_nom_opt_GW", "kwak_full（最佳化後）", "#dd8452"),
    ("BPLE 官方_GW（限制檔）", "BPLE 官方目標（限制檔）", "#8c8c8c"),
    ("論文 Table2_GW", "論文 Table 2", "none"),
]
x = np.arange(len(pc))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
for j, (col, lab, colr) in enumerate(series):
    offset = (j - 1.5) * width
    if colr == "none":
        ax.bar(x + offset, pc[col], width, facecolor="none",
               edgecolor="#333333", linewidth=1.4, label=lab)
    else:
        ax.bar(x + offset, pc[col], width, color=colr, edgecolor="white", label=lab)

ax.set_xticks(x)
ax.set_xticklabels(pc["中文"], rotation=15)
ax.set_ylabel("裝置容量 [GW]")
ax.set_title("KR2036 裝置容量驗證：模型 vs BPLE 官方目標 vs 論文 Table 2", fontsize=13, pad=10)
ax.legend(frameon=False, ncol=4)
ax.grid(axis="y", alpha=0.3)

K.savefig("04_fig1_capacity_validation")
plt.show()

### 3.1 容量驗證結論

- **核能 31.7 GW**：三方完全一致（BPLE = 論文 = 模型），因為限制檔把它釘死。
- **CCGT**：模型 64.6 GW（基準）與 62.9 GW（kwak_full）分別對準 BPLE 與論文，各自吻合。
- **燃煤**：BPLE 與論文都是 27.1 GW，但**基準情境是 31.2 GW**——限制檔裡沒有 coal 這一列，
  燃煤容量是由既有電廠資料（`powerplants.csv`）決定而非受限，因此高出 4.1 GW。
  kwak_full 已改用論文的 27.1 GW 固定值。
- **生質能**：模型頂到 CCL 區間上限 1.98 GW，論文固定 1.62 GW。`docs/比較分析…md`
  第 4 節已載明這是「我們仍設 CCL 區間而非固定值」造成的，屬已知差異。
- **太陽光電**：基準 72.3 GW 頂到 BPLE 上限（65.7 +10%），kwak_full 68.1 GW 落在區間內部。
- **風電合計**：基準 30.7 GW 貼近 BPLE 下限，kwak_full 64.7 GW 採論文值——
  **論文的風電假設是 BPLE 官方目標（34.1 GW）的近兩倍**，這是兩個情境最大的容量差異來源。

## 4. 發電佔比驗證

對照 `table3_generation_opex.csv` 的 `paper_generation_TWh` 與 `paper_share_pct`。
模型發電量用 `snapshot_weightings` 加權計算，已排除 load shedding 虛擬機組。

In [ ]:
rows = []
for carrier in ["nuclear", "coal", "CCGT", "biomass", "solar", "onwind", "offwind-ac", "offwind-dc", "ror", "oil"]:
    r = {"carrier": carrier, "中文": K.carrier_label(carrier)}
    for key, m in NETS.items():
        idx = m.generators.index[m.generators.carrier == carrier]
        w = m.snapshot_weightings.generators
        r[f"{K.SCENARIOS[key]['label']}_TWh"] = float(
            m.generators_t.p[idx].mul(w, axis=0).sum().sum() / 1e6
        ) if len(idx) else 0.0
    paper_key = "offwind" if carrier.startswith("offwind") else carrier
    r["論文_TWh"] = float(t3.at[paper_key, "paper_generation_TWh"]) if paper_key in t3.index else np.nan
    rows.append(r)

gen = pd.DataFrame(rows).set_index("carrier")

# 離岸風在論文是合計，補一列可比的
for key, m in NETS.items():
    idx = m.generators.index[m.generators.carrier.isin(["offwind-ac", "offwind-dc"])]
    w = m.snapshot_weightings.generators
    gen.loc["offwind合計", f"{K.SCENARIOS[key]['label']}_TWh"] = float(
        m.generators_t.p[idx].mul(w, axis=0).sum().sum() / 1e6
    )
gen.loc["offwind合計", "中文"] = "離岸風電合計"
gen.loc["offwind合計", "論文_TWh"] = float(t3.at["offwind", "paper_generation_TWh"])

# 佔比（以各自的總發電為分母）
for key in NETS:
    col = f"{K.SCENARIOS[key]['label']}_TWh"
    base = gen.loc[[i for i in gen.index if i not in ("offwind合計",)], col].sum()
    gen[f"{K.SCENARIOS[key]['label']}_佔比%"] = gen[col] / base * 100
gen["論文_佔比%"] = gen.index.map(
    lambda i: float(t3.at["offwind" if i.startswith("offwind") else i, "paper_share_pct"])
    if ("offwind" if i.startswith("offwind") else i) in t3.index
    and not pd.isna(t3.at["offwind" if i.startswith("offwind") else i, "paper_share_pct"])
    else np.nan
)
gen["kwak_full 對論文偏差%"] = (gen["kwak_full_TWh"] / gen["論文_TWh"] - 1) * 100

gen.round(3).to_csv(K.FIG_DIR / "04_table_generation_validation.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "04_table_generation_validation.csv")
gen.round(2)

In [ ]:
plot_rows = ["nuclear", "coal", "CCGT", "biomass", "solar", "onwind", "offwind合計"]
pg = gen.loc[plot_rows]
x = np.arange(len(pg))
width = 0.26

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

ax = axes[0]
ax.bar(x - width, pg["基準情境_TWh"], width, color="#4c72b0", edgecolor="white", label="基準情境")
ax.bar(x, pg["kwak_full_TWh"], width, color="#dd8452", edgecolor="white", label="kwak_full")
ax.bar(x + width, pg["論文_TWh"], width, facecolor="none", edgecolor="#333333",
       linewidth=1.4, label="論文 Table 3")
ax.set_xticks(x)
ax.set_xticklabels(pg["中文"], rotation=20)
ax.set_ylabel("年發電量 [TWh]")
ax.set_title("發電量對照", fontsize=12)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
dev = pg["kwak_full 對論文偏差%"]
# "offwind合計" 不是 config 裡的 carrier，明確指定顏色避免掉進灰色 fallback
bar_colors = [K.carrier_color("offwind-dc" if i == "offwind合計" else i) for i in pg.index]
ax.barh(pg["中文"], dev, color=bar_colors, edgecolor="white")
ax.axvline(0, color="#444", linewidth=0.9)
for band in (15, -15):
    ax.axvline(band, color="#999", linestyle=":", linewidth=1.0)
ax.set_xlabel("kwak_full 相對論文的偏差 [%]")
ax.set_title("偏差幅度（虛線為 ±15%）", fontsize=12)
ax.grid(axis="x", alpha=0.3)

span = float(dev.abs().max())
ax.set_xlim(-span * 1.35, span * 1.35)  # 兩側留白，數字標籤才不會被切掉或壓到軸標籤
for i, v in enumerate(dev):
    pad = span * 0.03
    ax.text(v + (pad if v >= 0 else -pad), i, f"{v:+.1f}%", va="center",
            ha="left" if v >= 0 else "right", fontsize=8)

fig.suptitle("KR2036 發電量驗證：模型 vs 論文 Table 3", fontsize=14)
K.savefig("04_fig2_generation_validation")
plt.show()

### 4.1 發電驗證結論

以 kwak_full 對論文 Table 3 判讀（±15% 內視為重現）：

- **落在 ±15% 內**：核能（+1.3%）、CCGT（+3.5%）、燃煤（−14.1%）、太陽光電（+12.5%）。
  這四項合計超過總發電的四分之三，代表**在相同假設下工具重現了論文的調度結構**。
- **明顯偏離**：陸域風電 **−41%**、離岸風電 **+24%**、生質能 **+133%**。
  前兩者依 `docs/比較分析…md` 第 4 節的歸因，是**節點解析度造成的選址差異**
  （論文 120 節點能把陸域風配置在東部山區好站點，10 節點把整區平均掉；
  離岸風則相反）。風電**總量**差異只有約 4%，但陸海比例不同。
  生質能偏高是因為容量頂到 CCL 上限且碳約束較緊。
- **不做結論的項目**：總成本與平均電價，`table0_assumptions.csv` 已載明口徑不同。

**基準情境的核能為什麼是 220 TWh（對論文 +37%）**：兩個情境的核能可用率不同——
基準情境 `p_max_pu = 0.80`，kwak_full 才是論文反推的 `0.60`（見下方驗證）。
基準情境的核能幾乎整年頂在 0.80 上限運轉（CF 0.79），所以發電量高出許多。
**這不是誤差，是情境設定差異**；要對論文做驗證請一律看 kwak_full 欄。

In [ ]:
# 核能可用率設定差異（解釋上面兩情境核能發電量的落差）
rows = []
for key, m in NETS.items():
    g = m.generators
    nuc = g[g.carrier == "nuclear"]
    w = m.snapshot_weightings.generators
    gen_twh = float(m.generators_t.p[nuc.index].mul(w, axis=0).sum().sum() / 1e6)
    cap_gw = float(nuc.p_nom_opt.sum() / 1e3)
    rows.append({
        "情境": K.SCENARIOS[key]["label"],
        "核能容量_GW": cap_gw,
        "p_max_pu（可用率）": float(nuc.p_max_pu.mean()),
        "發電_TWh": gen_twh,
        "實際 CF": gen_twh * 1e6 / (cap_gw * 1e3 * 8760),
    })
nuclear_chk = pd.DataFrame(rows).set_index("情境")
nuclear_chk.loc["論文 Table 3"] = {
    "核能容量_GW": float(t2.at["nuclear", "paper_capacity_MW"]) / 1e3,
    "p_max_pu（可用率）": np.nan,
    "發電_TWh": float(t3.at["nuclear", "paper_generation_TWh"]),
    "實際 CF": float(t3.at["nuclear", "paper_generation_TWh"]) * 1e6
    / (float(t2.at["nuclear", "paper_capacity_MW"]) * 8760),
}
nuclear_chk.round(3)

## 5. 已移除的原版驗證項目

原版的三個外部資料來源全部移除，理由如下：

| 原版來源 | 移除理由 |
|---|---|
| **IRENA** 再生能源容量統計 | 涵蓋的是歷史實績年份，本研究是 2036 年情境，無可對照的目標值 |
| **USAID** 電力部門資料 | 僅涵蓋非洲與部分開發中國家，不含韓國 |
| **Our World in Data** 發電結構 | 同為歷史實績（原版取 2020 年），且需連外下載；韓國 2036 的對照值改用 BPLE 與論文 |

取而代之的是 repo 內既有的 BPLE 官方目標與論文 Table 2/3，全部可離線重現。

**尚無對照值的項目**：本 notebook 沒有納入輸電線路長度的官方對照
（見 01 第 1.2.6 節，KEPCO 統計不在 repo 內），也沒有納入實際 2036 年的實績
（尚未發生）。這些在補進資料前不會有數字。